# 模型量化教程 (Model Quantization Tutorial)

> **前置知识**: PyTorch 基础、深度学习模型训练流程、基本数学知识
>
> **学习目标**: 掌握量化的数学原理、实现方法和最佳实践

---

## 为什么需要量化？

```
┌─────────────────────────────────────────────────────────────┐
│                    量化的核心价值                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  FP32 模型                        INT8 模型                 │
│  ┌─────────────┐                  ┌─────────────┐           │
│  │ 4 字节/参数 │                  │ 1 字节/参数 │           │
│  │ 高精度计算  │      量化        │ 整数计算    │           │
│  │ 内存占用大  │  ──────────→     │ 内存减少 4x │           │
│  │ 计算较慢    │                  │ 计算加速 2-4x│           │
│  └─────────────┘                  └─────────────┘           │
│                                                             │
│  关键洞察: 神经网络对精度损失有一定容忍度                   │
│           适度降低精度不会显著影响模型效果                  │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 数据类型对比

| 数据类型 | 位数 | 内存占用 | 数值范围 | 计算速度 | 适用场景 |
|:---------|:----:|:--------:|:---------|:--------:|:---------|
| FP32 | 32 | 4 字节 | ±3.4×10³⁸ | 基准 | 训练、高精度推理 |
| FP16 | 16 | 2 字节 | ±65504 | 2x 加速 | GPU 混合精度 |
| INT8 | 8 | 1 字节 | [-128, 127] | 2-4x 加速 | 服务器部署 |
| INT4 | 4 | 0.5 字节 | [-8, 7] | 4-8x 加速 | LLM 推理 |

## 本教程内容

1. **量化基础** - 量化公式、scale 和 zero_point 的计算
2. **对称 vs 非对称量化** - 两种量化方式的对比
3. **动态量化** - 推理时动态计算量化参数
4. **静态量化** - 使用校准数据预计算量化参数
5. **量化感知训练 (QAT)** - 在训练中模拟量化效果

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time

# 设置随机种子，确保结果可复现
torch.manual_seed(42)
np.random.seed(42)

print("=" * 50)
print("环境准备完成")
print("=" * 50)
print(f"PyTorch 版本: {torch.__version__}")

## 1. 量化基础

**核心概念**: 量化是将连续或高精度的数值映射到离散或低精度数值的过程

```
┌─────────────────────────────────────────────────────────────┐
│                    量化过程示意图                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  浮点数 (连续)                    整数 (离散)               │
│  ─────────────                    ───────────               │
│  -2.5 ─────────┐                                            │
│  -1.8 ─────────┼──→ 量化 ──→     -2                        │
│  -1.2 ─────────┘                                            │
│                                                             │
│   0.3 ─────────┐                                            │
│   0.5 ─────────┼──→ 量化 ──→      0                        │
│   0.8 ─────────┘                                            │
│                                                             │
│   2.1 ─────────┐                                            │
│   2.7 ─────────┼──→ 量化 ──→      2                        │
│   3.0 ─────────┘                                            │
│                                                             │
│  多个浮点值映射到同一个整数，产生量化误差                   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 1.1 量化公式

**量化过程 (浮点 → 整数)**:
```
q = round(r / scale) + zero_point
q = clamp(q, qmin, qmax)  # 限制在整数范围内
```

**反量化过程 (整数 → 浮点)**:
```
r' = scale × (q - zero_point)
```

**参数说明**:
- `r`: 原始浮点值
- `q`: 量化后的整数值
- `scale`: 缩放因子，决定量化精度
- `zero_point`: 零点，使浮点 0 映射到整数 zero_point

In [ ]:
# ============================================================
# 量化参数计算
# ============================================================

def compute_scale_zero_point(r_min, r_max, qmin, qmax, symmetric=True):
    """
    计算量化参数 scale 和 zero_point
    
    参数:
        r_min: 浮点数最小值
        r_max: 浮点数最大值
        qmin: 量化整数最小值 (如 -128 for INT8)
        qmax: 量化整数最大值 (如 127 for INT8)
        symmetric: 是否使用对称量化
        
    返回:
        scale: 缩放因子
        zero_point: 零点
        
    计算公式:
    ┌─────────────────────────────────────────────────────────┐
    │  对称量化:                                              │
    │    scale = max(|r_min|, |r_max|) / qmax                │
    │    zero_point = 0                                      │
    │                                                         │
    │  非对称量化:                                            │
    │    scale = (r_max - r_min) / (qmax - qmin)             │
    │    zero_point = round(qmin - r_min / scale)            │
    └─────────────────────────────────────────────────────────┘
    """
    if symmetric:
        # 对称量化: zero_point = 0
        # 使用绝对值最大的范围，确保 0 映射到 0
        abs_max = max(abs(r_min), abs(r_max))
        scale = abs_max / qmax
        zero_point = 0
    else:
        # 非对称量化: 充分利用整数范围
        scale = (r_max - r_min) / (qmax - qmin)
        zero_point = int(round(qmin - r_min / scale))
        # 确保 zero_point 在有效范围内
        zero_point = max(qmin, min(qmax, zero_point))
    
    return scale, zero_point


def quantize_tensor(x, scale, zero_point, qmin, qmax):
    """
    量化张量: 浮点 → 整数
    
    q = clamp(round(x / scale) + zero_point, qmin, qmax)
    """
    q = torch.round(x / scale) + zero_point
    q = torch.clamp(q, qmin, qmax)
    return q.to(torch.int8)


def dequantize_tensor(q, scale, zero_point):
    """
    反量化张量: 整数 → 浮点
    
    r' = scale × (q - zero_point)
    """
    return scale * (q.float() - zero_point)


print("量化函数定义完成!")

In [ ]:
# ============================================================
# 量化效果演示
# ============================================================
print("=" * 60)
print("量化效果演示")
print("=" * 60)

# 创建示例数据: 1000 个服从正态分布的浮点数
x = torch.randn(1000) * 2  # 范围约 [-6, 6]

# 计算量化参数
x_min, x_max = x.min().item(), x.max().item()
scale, zero_point = compute_scale_zero_point(
    x_min, x_max, 
    qmin=-128, qmax=127,  # INT8 范围
    symmetric=True
)

print(f"\n原始数据统计:")
print(f"  范围: [{x_min:.4f}, {x_max:.4f}]")
print(f"  均值: {x.mean():.4f}")
print(f"  标准差: {x.std():.4f}")

print(f"\n量化参数:")
print(f"  Scale: {scale:.6f}")
print(f"  Zero Point: {zero_point}")
print(f"  量化精度: {scale:.6f} (每个整数单位代表的浮点值)")

In [ ]:
# ============================================================
# 量化和反量化
# ============================================================

# 执行量化
x_quantized = quantize_tensor(x, scale, zero_point, -128, 127)

# 执行反量化
x_dequantized = dequantize_tensor(x_quantized, scale, zero_point)

# 计算量化误差
error = (x - x_dequantized).abs()

print("=" * 60)
print("量化结果分析")
print("=" * 60)

print(f"\n数据类型变化:")
print(f"  原始: {x.dtype} → 量化后: {x_quantized.dtype}")

print(f"\n量化值范围:")
print(f"  [{x_quantized.min().item()}, {x_quantized.max().item()}]")

print(f"\n量化误差统计:")
print(f"  平均误差: {error.mean():.6f}")
print(f"  最大误差: {error.max():.6f}")
print(f"  相对误差: {(error / (x.abs() + 1e-8)).mean() * 100:.2f}%")

print(f"\n内存节省:")
print(f"  原始大小: {x.numel() * 4} 字节 (FP32)")
print(f"  量化大小: {x_quantized.numel() * 1} 字节 (INT8)")
print(f"  压缩比: 4x")

In [ ]:
# ============================================================
# 可视化量化效果
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 图1: 原始数据分布
axes[0].hist(x.numpy(), bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_title('原始数据分布 (FP32)', fontsize=12)
axes[0].set_xlabel('值')
axes[0].set_ylabel('频数')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)

# 图2: 量化后数据分布
axes[1].hist(x_quantized.numpy(), bins=50, alpha=0.7, color='green', edgecolor='black')
axes[1].set_title('量化后数据分布 (INT8)', fontsize=12)
axes[1].set_xlabel('量化值')
axes[1].set_ylabel('频数')
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

# 图3: 量化误差分布
axes[2].hist(error.numpy(), bins=50, alpha=0.7, color='red', edgecolor='black')
axes[2].set_title('量化误差分布', fontsize=12)
axes[2].set_xlabel('绝对误差')
axes[2].set_ylabel('频数')

plt.tight_layout()
plt.show()

print("\n观察: 量化误差很小，大部分误差集中在 scale/2 附近")

## 2. 对称量化 vs 非对称量化

**核心区别**: zero_point 是否为 0

```
┌─────────────────────────────────────────────────────────────┐
│                    对称量化 (Symmetric)                      │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  特点:                                                      │
│  - zero_point = 0                                          │
│  - 浮点 0 精确映射到整数 0                                 │
│  - 量化范围: [-127, 127] (保留对称性)                      │
│                                                             │
│  浮点范围        量化范围                                   │
│  [-3.0, 3.0] → [-127, 127]                                 │
│       0      →      0                                       │
│                                                             │
│  优点: 计算简单，无需存储 zero_point                        │
│  缺点: 对非对称分布的数据效率较低                          │
│  适用: 权重 (通常以 0 为中心分布)                          │
│                                                             │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                   非对称量化 (Asymmetric)                    │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  特点:                                                      │
│  - zero_point ≠ 0                                          │
│  - 充分利用整数范围                                        │
│  - 量化范围: [0, 255] 或 [-128, 127]                       │
│                                                             │
│  浮点范围        量化范围                                   │
│  [0.0, 6.0]  → [0, 255]                                    │
│      0       →    0                                         │
│      6       →   255                                        │
│                                                             │
│  优点: 更好地利用量化范围，精度更高                         │
│  缺点: 需要额外存储 zero_point                             │
│  适用: 激活值 (如 ReLU 输出，范围 [0, +∞))                 │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 对称 vs 非对称量化对比
# ============================================================
print("=" * 60)
print("对称 vs 非对称量化对比")
print("=" * 60)

# 模拟 ReLU 输出 (非负值，典型的非对称分布)
x_relu = F.relu(torch.randn(1000) * 2)

print(f"\nReLU 输出数据:")
print(f"  范围: [{x_relu.min():.4f}, {x_relu.max():.4f}]")
print(f"  均值: {x_relu.mean():.4f}")

# ============================================================
# 对称量化
# ============================================================
scale_sym, zp_sym = compute_scale_zero_point(
    x_relu.min().item(), x_relu.max().item(), 
    -128, 127, 
    symmetric=True
)
x_q_sym = quantize_tensor(x_relu, scale_sym, zp_sym, -128, 127)
x_dq_sym = dequantize_tensor(x_q_sym, scale_sym, zp_sym)
error_sym = (x_relu - x_dq_sym).abs().mean()

print(f"\n对称量化:")
print(f"  Scale: {scale_sym:.6f}")
print(f"  Zero Point: {zp_sym}")
print(f"  量化值范围: [{x_q_sym.min().item()}, {x_q_sym.max().item()}]")
print(f"  平均误差: {error_sym:.6f}")
print(f"  问题: 负数范围 [-128, 0) 完全浪费!")

# ============================================================
# 非对称量化
# ============================================================
scale_asym, zp_asym = compute_scale_zero_point(
    x_relu.min().item(), x_relu.max().item(), 
    0, 255,  # 使用无符号整数范围
    symmetric=False
)
x_q_asym = quantize_tensor(x_relu, scale_asym, zp_asym, 0, 255)
x_dq_asym = dequantize_tensor(x_q_asym, scale_asym, zp_asym)
error_asym = (x_relu - x_dq_asym).abs().mean()

print(f"\n非对称量化:")
print(f"  Scale: {scale_asym:.6f}")
print(f"  Zero Point: {zp_asym}")
print(f"  量化值范围: [{x_q_asym.min().item()}, {x_q_asym.max().item()}]")
print(f"  平均误差: {error_asym:.6f}")
print(f"  优势: 充分利用 [0, 255] 范围!")

# ============================================================
# 结论
# ============================================================
print(f"\n" + "=" * 60)
print("结论")
print("=" * 60)
improvement = (error_sym - error_asym) / error_sym * 100
print(f"非对称量化误差降低: {improvement:.1f}%")
print(f"\n建议:")
print(f"  - 权重: 使用对称量化 (通常以 0 为中心)")
print(f"  - 激活值: 使用非对称量化 (如 ReLU 输出)")

## 3. 动态量化

**核心概念**: 动态量化在推理时动态计算激活值的量化参数，权重预先量化

```
┌─────────────────────────────────────────────────────────────┐
│                    动态量化流程                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  训练阶段:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  正常训练 FP32 模型                                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  量化阶段:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 权重: 预先量化为 INT8                           │   │
│  │  2. 激活: 保持 FP32，推理时动态量化                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  推理阶段:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  输入 (FP32) → 动态计算 scale/zp → 量化 → 计算     │   │
│  │                                    ↓                │   │
│  │                              INT8 矩阵乘法          │   │
│  │                                    ↓                │   │
│  │                              反量化 → 输出 (FP32)   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  优点: 无需校准数据，实现简单                               │
│  缺点: 每次推理都需计算量化参数，有额外开销                 │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 定义测试模型
# ============================================================

class SimpleClassifier(nn.Module):
    """
    简单分类器模型
    
    结构: Linear → ReLU → Linear → ReLU → Linear
    用于演示量化效果
    """
    def __init__(self, input_dim=784, hidden_dim=256, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim // 2, num_classes)
    
    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.fc3(x)


def get_model_size(model):
    """计算模型大小 (MB)"""
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    return param_size / (1024 * 1024)


# 创建模型
model = SimpleClassifier()
model.eval()

print("=" * 60)
print("测试模型信息")
print("=" * 60)
print(f"\n模型结构:")
print(f"  输入: 784 → 隐藏: 256 → 隐藏: 128 → 输出: 10")
print(f"\n参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"模型大小: {get_model_size(model):.2f} MB")

In [ ]:
# ============================================================
# 应用动态量化
# ============================================================
print("=" * 60)
print("动态量化演示")
print("=" * 60)

# PyTorch 动态量化 API
# 只量化 Linear 层，这是最常见的做法
quantized_model = torch.quantization.quantize_dynamic(
    model,                      # 原始模型
    {nn.Linear},                # 要量化的层类型
    dtype=torch.qint8           # 量化数据类型
)

print(f"\n量化完成!")
print(f"\n量化后的模型结构:")
print(quantized_model)

In [ ]:
# ============================================================
# 比较原始模型和量化模型
# ============================================================

# 测试数据
x_test = torch.randn(32, 784)

# 推理
with torch.no_grad():
    original_output = model(x_test)
    quantized_output = quantized_model(x_test)

# 比较输出
output_diff = (original_output - quantized_output).abs()

print("=" * 60)
print("输出比较")
print("=" * 60)
print(f"\n输出差异:")
print(f"  平均差异: {output_diff.mean():.6f}")
print(f"  最大差异: {output_diff.max():.6f}")

# 检查预测是否一致
original_pred = original_output.argmax(dim=1)
quantized_pred = quantized_output.argmax(dim=1)
accuracy = (original_pred == quantized_pred).float().mean()
print(f"\n预测一致率: {accuracy * 100:.1f}%")
print(f"\n结论: 量化后预测结果几乎完全一致!")

In [ ]:
# ============================================================
# 性能基准测试
# ============================================================

def benchmark_model(model, input_tensor, num_runs=100, warmup=10):
    """
    测量模型推理时间
    
    参数:
        model: 待测试模型
        input_tensor: 输入张量
        num_runs: 测试次数
        warmup: 预热次数
    """
    model.eval()
    
    # 预热 (让 CPU 缓存稳定)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_tensor)
    
    # 计时
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = model(input_tensor)
            end = time.perf_counter()
            times.append((end - start) * 1000)  # 转换为毫秒
    
    return np.mean(times), np.std(times)


# 基准测试
x_bench = torch.randn(64, 784)

print("=" * 60)
print("性能基准测试")
print("=" * 60)
print(f"\n测试配置: batch_size=64, 100 次运行")

original_time, original_std = benchmark_model(model, x_bench)
quantized_time, quantized_std = benchmark_model(quantized_model, x_bench)

print(f"\n推理时间:")
print(f"  原始模型:   {original_time:.3f} ± {original_std:.3f} ms")
print(f"  量化模型:   {quantized_time:.3f} ± {quantized_std:.3f} ms")
print(f"\n加速比: {original_time / quantized_time:.2f}x")
print(f"\n注意: CPU 上的加速效果取决于硬件支持")

## 4. 静态量化

**核心概念**: 静态量化使用校准数据预先计算所有层的量化参数

```
┌─────────────────────────────────────────────────────────────┐
│                    静态量化流程                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 准备校准数据集 (代表性数据，通常 100-1000 样本)        │
│                        ↓                                    │
│  2. 运行模型，收集每层激活值的统计信息                      │
│     ┌─────────────────────────────────────────────────┐    │
│     │  Layer 1: min=-2.3, max=4.5                     │    │
│     │  Layer 2: min=-1.1, max=3.2                     │    │
│     │  ...                                            │    │
│     └─────────────────────────────────────────────────┘    │
│                        ↓                                    │
│  3. 计算每层的 scale 和 zero_point                          │
│                        ↓                                    │
│  4. 量化权重和激活值                                        │
│                        ↓                                    │
│  5. 导出量化模型                                            │
│                                                             │
│  优点: 推理时无需动态计算，速度更快                         │
│  缺点: 需要校准数据，校准数据质量影响精度                   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 静态量化 vs 动态量化

| 特性 | 动态量化 | 静态量化 |
|:-----|:---------|:---------|
| 校准数据 | 不需要 | 需要 |
| 量化参数计算 | 推理时动态计算 | 预先计算 |
| 推理速度 | 较快 | 最快 |
| 精度 | 中等 | 较高 |
| 适用场景 | 快速部署 | 生产环境 |

In [ ]:
# ============================================================
# 静态量化实现
# ============================================================

def calibrate_model(model, calibration_data, num_batches=30):
    """
    校准模型，收集激活值统计信息
    
    参数:
        model: 待校准模型
        calibration_data: 校准数据列表
        num_batches: 使用的批次数
        
    返回:
        activation_stats: 每层的激活值统计 {layer_name: {min: [], max: []}}
    """
    activation_stats = {}
    hooks = []
    
    def hook_fn(name):
        """创建 hook 函数，记录激活值范围"""
        def hook(module, input, output):
            if name not in activation_stats:
                activation_stats[name] = {'min': [], 'max': []}
            activation_stats[name]['min'].append(output.min().item())
            activation_stats[name]['max'].append(output.max().item())
        return hook
    
    # 注册 hook 到需要量化的层
    for name, module in model.named_modules():
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            hooks.append(module.register_forward_hook(hook_fn(name)))
    
    # 运行校准数据
    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(calibration_data):
            if i >= num_batches:
                break
            model(batch)
    
    # 移除 hooks
    for hook in hooks:
        hook.remove()
    
    return activation_stats


# 创建新模型用于静态量化
model_static = SimpleClassifier()
model_static.eval()

# 创建校准数据 (模拟真实数据分布)
calibration_data = [torch.randn(32, 784) for _ in range(50)]

print("=" * 60)
print("静态量化 - 校准阶段")
print("=" * 60)

# 校准
print("\n正在校准模型...")
activation_stats = calibrate_model(model_static, calibration_data, num_batches=30)

print(f"\n收集到 {len(activation_stats)} 层的激活值统计:")
for name, stats in activation_stats.items():
    min_val = min(stats['min'])
    max_val = max(stats['max'])
    print(f"  {name}: min={min_val:.4f}, max={max_val:.4f}")

## 5. 量化感知训练 (QAT)

**核心概念**: QAT 在训练过程中模拟量化效果，使模型学习适应量化误差

```
┌─────────────────────────────────────────────────────────────┐
│                    QAT vs PTQ 对比                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  PTQ (训练后量化):                                          │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  训练 (FP32) → 量化 → 推理 (INT8)                   │   │
│  │  模型不知道会被量化，可能对量化误差敏感             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  QAT (量化感知训练):                                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  训练时模拟量化 → 模型适应量化误差 → 推理 (INT8)    │   │
│  │  模型在训练中学会容忍量化误差，精度更高             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 伪量化 (Fake Quantization)

**关键技术**: 前向传播模拟量化，反向传播使用直通估计器 (STE)

```
┌─────────────────────────────────────────────────────────────┐
│                    为什么需要 STE？                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  问题: round() 函数的梯度几乎处处为 0                       │
│                                                             │
│        y = round(x)                                         │
│        dy/dx = 0 (几乎处处)                                 │
│                                                             │
│  如果直接计算梯度，模型无法学习！                           │
│                                                             │
│  解决方案: 直通估计器 (STE)                                 │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  前向: y = round(x)                                 │   │
│  │  反向: dy/dx ≈ 1 (假装 round 不存在)                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  直觉: 虽然不精确，但允许梯度流动，模型可以学习             │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 伪量化实现 (Fake Quantization)
# ============================================================

class FakeQuantize(torch.autograd.Function):
    """
    伪量化 - 前向量化，反向直通
    
    前向传播: 量化 → 反量化 (模拟量化误差)
    反向传播: 梯度直接传递 (忽略量化操作)
    
    这是 QAT 的核心技术!
    """
    
    @staticmethod
    def forward(ctx, x, scale, zero_point, qmin, qmax):
        """
        前向传播: 模拟量化效果
        
        1. 量化: 浮点 → 整数
        2. 反量化: 整数 → 浮点
        3. 返回带有量化误差的浮点值
        """
        # 量化
        q = torch.round(x / scale + zero_point)
        q = torch.clamp(q, qmin, qmax)
        # 反量化 (引入量化误差)
        x_q = (q - zero_point) * scale
        return x_q
    
    @staticmethod
    def backward(ctx, grad_output):
        """
        反向传播: 直通估计器 (STE)
        
        梯度直接传递，不经过量化操作
        这允许模型学习，尽管量化是不可微的
        """
        # 梯度直接传递
        return grad_output, None, None, None, None


# 演示伪量化
print("=" * 60)
print("伪量化演示")
print("=" * 60)

x = torch.randn(10, requires_grad=True)
scale = torch.tensor(0.1)
zero_point = torch.tensor(0.0)

# 伪量化
x_fake_q = FakeQuantize.apply(x, scale, zero_point, -128, 127)

print(f"\n原始值 (前5个): {x[:5].detach().numpy().round(4)}")
print(f"伪量化后 (前5个): {x_fake_q[:5].detach().numpy().round(4)}")

# 验证梯度可以传播
loss = x_fake_q.sum()
loss.backward()
print(f"\n梯度 (前5个): {x.grad[:5].numpy()}")
print("\n✓ 梯度成功传播! (STE 工作正常)")

## 6. 总结

### 量化方法对比

| 方法 | 校准数据 | 训练 | 精度 | 速度 | 适用场景 |
|:-----|:--------:|:----:|:----:|:----:|:---------|
| 动态量化 | 不需要 | 不需要 | 中 | 快 | 快速部署、原型验证 |
| 静态量化 | 需要 | 不需要 | 高 | 最快 | 生产环境部署 |
| QAT | 需要 | 需要 | 最高 | 最快 | 精度敏感场景 |

### 选择建议

```
┌─────────────────────────────────────────────────────────────┐
│                    量化方法选择指南                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Q1: 是否需要快速部署？                                     │
│  ├── 是 → 动态量化 (无需校准数据)                          │
│  └── 否 → 继续 Q2                                          │
│                                                             │
│  Q2: 是否有校准数据？                                       │
│  ├── 否 → 动态量化                                         │
│  └── 是 → 继续 Q3                                          │
│                                                             │
│  Q3: 精度要求是否很高？                                     │
│  ├── 是 → QAT (量化感知训练)                               │
│  └── 否 → 静态量化 (PTQ)                                   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 最佳实践

```
量化检查清单:
✓ 权重使用对称量化 (通常以 0 为中心)
✓ 激活值使用非对称量化 (如 ReLU 输出)
✓ 校准数据要有代表性 (100-1000 样本)
✓ 量化后验证模型精度
✓ 在目标硬件上测试性能

常见陷阱:
✗ 校准数据分布与实际数据不一致
✗ 忽略量化敏感层 (如第一层、最后一层)
✗ 没有在目标硬件上验证加速效果
```

### 下一步学习

- **02_Pruning_tutorial.ipynb**: 模型剪枝技术
- **03_Distillation_tutorial.ipynb**: 知识蒸馏
- **04_Export_tutorial.ipynb**: 模型导出与部署